# Four-body system with multirate potential

Author: Kevin Schäfers, kschaefers@uni-wuppertal.de

Date: 08.04.2026

---------------------------------------------------

This Jupyter notebook implements the hierarchical splitting methods used in 

*K. Schäfers, M. Günther: A hierarchical splitting approach for N-split ordinary differential equations*

to perform the numerical simulations for the four-body problem from [Hairer, Lubich, Wanner: Geometric Numerical Integration, Example VIII.4.4]. Moreover, this Jupyter notebook contains the scripts with which the numerical results have been obtained. For the numerical results reported in the paper, we have executed this Jupyter notebook with Python version 3.13.0 on a MacBook Pro 2021 with M1 Pro chip.

In [ ]:
import numpy as np
import time 
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp # for computing the reference solution
import matplot2tikz # use matplot2tikz.save() to save the figure as a .tex file for use in LaTeX documents

Initially, we define the Hamiltonian and the corresponding equations of motion (already paritioned into three subsystems).

In [ ]:
m = np.array([1,1e-2,1e-2,1e-4]) # masses of the planets and the satellite

Minv = np.linalg.inv(np.diag([m[0],m[0],m[1],m[1],m[2],m[2],m[3],m[3]])) # inverse mass matrix used for the kinetic energy

def H(p,q):
    """Hamiltonian of the N-body problem, consisting of kinetic and potential energy."""
    tmp = 0.5 * np.transpose(p)@Minv@p 
    for j in range(4):
        for i in range(j):
            tmp -= m[i]*m[j]/np.linalg.norm(q[2*i:2*i+2]-q[2*j:2*j+2])

    return tmp
    
def dTdp(p):
    """Derivative of the kinetic energy."""
    return Minv@p

def f_slow(q):
    """Slow force of the N-body problem."""
    d = len(q)
    res = np.zeros(d)

    for j in range(4):
        for i in range(4):
            if i < j:
                if (i != 1 or j != 3):
                    diff = q[2*i:2*i+2]-q[2*j:2*j+2]
                    tmp = m[i]*m[j]*(diff)/np.linalg.norm(diff)**3
                    res[2*i:2*i+2] -= tmp
                    res[2*j:2*j+2] += tmp
    return res

def f_fast(q):
    """Fast force between the satellite and the first planet."""
    d = len(q)
    res = np.zeros(d)
    
    diff = q[2*1:2*1+2]-q[2*3:2*3+2]
    tmp = m[1]*m[3]*(diff)/np.linalg.norm(diff)**3
    res[2*1:2*1+2] -= tmp
    res[2*3:2*3+2] += tmp

    return res

We perform numerical simulations on the time interval $t \in [0,40]$ with initial values 

$$x_0 = (p_0,q_0) = (0,0,0,1e-2,0,0.5e-2,0,0,0,0,1,0,4,0,1.01,0).$$

In [ ]:
tspan = [0,40]
x0 = np.array([0,0,0,1e-2,0,0.5e-2,0,0,0,0,1,0,4,0,1.01,0]) # initial values x0 = (p0,q0)

In the next cell, we compute a reference solution via `scipy.integrate.solve_ivp` where the `RK45` method is used.

In [ ]:
def f(t,y):
    """Full right-hand side of the N-body problem."""
    p = y[:8]
    q = y[8:]
    dpdt = f_slow(q) + f_fast(q) 
    dqdt = dTdp(p)
    return np.concatenate((dpdt,dqdt))

sol = solve_ivp(
    f,
    tspan, x0,
    method="RK45",
    dense_output=True,
    rtol=2.3e-14, atol=1e-15
)  

x_eval = sol.sol(tspan[1]) # take the referece solution at the end of the time interval. Used to compute the global truncation error later on.

## Implementation of the hierarchical splitting methods 

We start with the implementation of the `HOMF4` method.

In [ ]:
def OMF_core_step(p,q,c,h,M,reweighting):
    """ 
    Inner step of the HOMF4 method which computes M time steps (with potential reweighting of the multirate factor) 
    of the six-stage splitting method from [Omelyan et al. 2003] for the kinetic part and the fast part of the potential.  

    Input:
    p : current state of the momenta 
    q : current state of the coordinates 
    c : fractional time step, i.e., the function aims to advance the numerical solution by a step of size c*h
    h : macro time step
    M : multirate factor
    reweighting: Boolean. If True, then the multirate factor is reweighted, otherwise a constant multirate factor M is utilized

    Output:
    p_new : updated momenta
    q_new : updated coordinates 
    """

    # integrator coefficients of the splitting method
    a2 = 0.253978510841060
    a3 = -0.032302867652700
    a4 = 1-2*(a2+a3)
    b1 = 0.083983152628767
    b2 = 0.682236533571909
    b3 = 0.5-(b1+b2)

    q_new = q.copy()
    p_new = p.copy()

    if reweighting: # potentially apply the reweighting of the multirate factor
        M = int(np.ceil(np.abs(c)*M))

    val = f_fast(q_new) # since the first momentum update of iteration m+1 uses the 
                        # same force evaluation as the last momentum update of iteration m, 
                        # we store the result of the force evaluation to re-use it.
    
    for m in range(M): # we compute M steps of the splitting method with step size c*h/M
        p_new += c*h*b1/M * val # momentum update
        q_new += c*h*a2/M * dTdp(p_new) # position update
        p_new += c*h*b2/M * f_fast(q_new) # ...
        q_new += c*h*a3/M * dTdp(p_new)
        p_new += c*h*b3/M * f_fast(q_new)
        q_new += c*h*a4/M * dTdp(p_new)
        p_new += c*h*b3/M * f_fast(q_new)
        q_new += c*h*a3/M * dTdp(p_new)
        p_new += c*h*b2/M * f_fast(q_new)
        q_new += c*h*a2/M * dTdp(p_new)
        val = f_fast(q_new)
        p_new += c*h*b1/M * val

    return p_new,q_new

def HierarchicalOMF4(x0,tspan,nsteps,M,reweighting=True):
    """ 
    HOMF4 method 
    ---
    Fourth-order hierarchical splitting method for separable Hamiltonian systems of the form H(p,q) = T(p) + V_S(q) + V_F(q).
    The method uses the six-stage splitting method from [Omelyan et al. 2003] in both inner nodes, and the exact flows in all leaf nodes.

    The implementation includes a multiple time stepping approach (incl. the possibility to apply a reweighting of the multirate factors 
    to reduce computational overhead).

    Input:
    x0 : initial value x_0 = (p_0,q_0) of the state consisting of the momenta p, and the generalized coordinates q
    tspan : time interval [t0,t_end] on which the numerical solution will be computed.
    nsteps : number of time steps of constant step size h=(tspan[1]-tspan[0])/nsteps, i.e., we consider an equidistant time grid 
    M : multirate factor -- the fast part of the potential and the kinetic part will be integrated by applying M steps of the splitting method 
    reweighting: Boolean -- If true, the multirate factor M will be reweighted. Otherwise, M is kept constant.

    Output:
    t : time grid 
    x : 2d array of size 2d x (nsteps+1) where x[:,n] contains the numerical approximation for the time point t[n]
    """

    h = (tspan[1] - tspan[0])/nsteps
    t = np.linspace(tspan[0], tspan[1], nsteps+1)
    x = np.zeros((len(x0), nsteps+1))
    x[:,0] = x0
    d = len(x0)//2

    # integrator coefficients
    a2 = 0.253978510841060
    a3 = -0.032302867652700
    a4 = 1-2*(a2+a3)
    b1 = 0.083983152628767
    b2 = 0.682236533571909
    b3 = 0.5-(b1+b2)
    
    val = f_slow(x[d:,0]) # since the first momentum update of iteration n+1 uses the 
                          # same force evaluation as the last momentum update of iteration n, 
                          # we store the result of the force evaluation to re-use it.

    for n in range(nsteps): 
        x[:,n+1] = x[:,n]
        x[:d,n+1] += h*b1*val # momentum update employing the slow part of the force
        x[:d,n+1],x[d:,n+1] = OMF_core_step(x[:d,n+1],x[d:,n+1],a2,h,M,reweighting) # fast subsystem computed by another call of the splitting method
        x[:d,n+1] += h*b2*f_slow(x[8:,n+1]) 
        x[:d,n+1],x[d:,n+1] = OMF_core_step(x[:d,n+1],x[d:,n+1],a3,h,M,reweighting)
        x[:d,n+1] += h*b3*f_slow(x[8:,n+1]) 
        x[:d,n+1],x[d:,n+1] = OMF_core_step(x[:d,n+1],x[d:,n+1],a4,h,M,reweighting)
        x[:d,n+1] += h*b3*f_slow(x[8:,n+1]) 
        x[:d,n+1],x[d:,n+1] = OMF_core_step(x[:d,n+1],x[d:,n+1],a3,h,M,reweighting)
        x[:d,n+1] += h*b2*f_slow(x[8:,n+1]) 
        x[:d,n+1],x[d:,n+1] = OMF_core_step(x[:d,n+1],x[d:,n+1],a2,h,M,reweighting)
        val = f_slow(x[d:,n+1]) 
        x[:d,n+1] += h*b1*val
        
    return t,x

Next, we implement the `COMP4` method that is only convergent of order $p=2$ but exhibits a computational order four for sufficiently large multirate factors.

In [ ]:
def Strang_core_step(p,q,c,h,M,reweighting):
    """ 
    Inner step of the COMP4 method which computes M time steps (with potential reweighting of the multirate factor) 
    of the Strang splitting for the kinetic part and the fast part of the potential.  

    Input:
    p : current state of the momenta 
    q : current state of the coordinates 
    c : fractional time step, i.e., the function aims to advance the numerical solution by a step of size c*h
    h : macro time step
    M : multirate factor
    reweighting: Boolean. If True, then the multirate factor is reweighted, otherwise a constant multirate factor M is utilized

    Output:
    p_new : updated momenta
    q_new : updated coordinates 
    """
    
    q_new = q.copy()
    p_new = p.copy()

    if reweighting: # potentially apply the reweighting of the multirate factor
        M = int(np.ceil(np.abs(c)*M))

    val = f_fast(q_new) # since the first momentum update of iteration m+1 uses the 
                        # same force evaluation as the last momentum update of iteration m, 
                        # we store the result of the force evaluation to re-use it.
    
    for m in range(M):
        p_new += 0.5*c*h/M * val # momentum update
        q_new += c*h/M * dTdp(p_new) # position update
        val = f_fast(q_new) 
        p_new += 0.5*c*h/M * val # momentum update

    return p_new,q_new

def COMP4(x0,tspan,nsteps,M,reweighting=True):
    """ 
    COMP4 method 
    ---
    Second-order hierarchical splitting method for separable Hamiltonian systems of the form H(p,q) = T(p) + V_S(q) + V_F(q) that, 
    for sufficiently large multirate factors M, exhibits computational order four.
    The method uses the six-stage splitting method from [Omelyan et al. 2003] in the root node, the Strang splitting in the second inner nodes,
    and the exact flows in all leaf nodes.

    Input:
    x0 : initial value x_0 = (p_0,q_0) of the state consisting of the momenta p, and the generalized coordinates q
    tspan : time interval [t0,t_end] on which the numerical solution will be computed.
    nsteps : number of time steps of constant step size h=(tspan[1]-tspan[0])/nsteps, i.e., we consider an equidistant time grid 
    M : multirate factor -- the fast part of the potential and the kinetic part will be integrated by applying M steps of the splitting method 
    reweighting: Boolean -- If true, the multirate factor M will be reweighted. Otherwise, M is kept constant.

    Output:
    t : time grid 
    x : 2d array of size 2d x (nsteps+1) where x[:,n] contains the numerical approximation for the time point t[n]
    """

    h = (tspan[1] - tspan[0])/nsteps
    t = np.linspace(tspan[0], tspan[1], nsteps+1)
    x = np.zeros((len(x0), nsteps+1))
    x[:,0] = x0
    d = len(x0)//2

    # integrator coefficients for the root node
    a2 = 0.253978510841060
    a3 = -0.032302867652700
    a4 = 1-2*(a2+a3)
    b1 = 0.083983152628767
    b2 = 0.682236533571909
    b3 = 0.5-(b1+b2)
    
    val = f_slow(x[8:,0]) # since the first momentum update of iteration n+1 uses the 
                          # same force evaluation as the last momentum update of iteration n, 
                          # we store the result of the force evaluation to re-use it.

    for n in range(nsteps):
        x[:,n+1] = x[:,n]
        x[:8,n+1] += h*b1*val # momentum update employing the slow part of the force
        x[:8,n+1],x[8:,n+1] = Strang_core_step(x[:8,n+1],x[8:,n+1],a2,h,M,reweighting) # fast subsystem computed via the Strang splitting
        x[:8,n+1] += h*b2*f_slow(x[8:,n+1])
        x[:8,n+1],x[8:,n+1] = Strang_core_step(x[:8,n+1],x[8:,n+1],a3,h,M,reweighting)
        x[:8,n+1] += h*b3*f_slow(x[8:,n+1]) 
        x[:8,n+1],x[8:,n+1] = Strang_core_step(x[:8,n+1],x[8:,n+1],a4,h,M,reweighting)
        x[:8,n+1] += h*b3*f_slow(x[8:,n+1]) 
        x[:8,n+1],x[8:,n+1] = Strang_core_step(x[:8,n+1],x[8:,n+1],a3,h,M,reweighting)
        x[:8,n+1] += h*b2*f_slow(x[8:,n+1]) 
        x[:8,n+1],x[8:,n+1] = Strang_core_step(x[:8,n+1],x[8:,n+1],a2,h,M,reweighting)
        val = f_slow(x[8:,n+1]) 
        x[:8,n+1] += h*b1*val
        
    return t,x

To compare the designed hierarchical splitting method with state-of-the-art methods, we implement the six-stage composition method of order four introduced in [Blanes & Moan 2002].

In [ ]:
def BMs6p4(x0,tspan,nsteps,M,reweighting=False):
    """ 
    Six-stage composition method of order four from [Blanes & Moan 2002] 
    ---
    Fourth-order splitting method with optimized coefficients as a method of RKN type.
    The method uses a composition of the Lie-Trotter splitting and its adjoint.

    The method is extended by a multiple time stepping approach (incl. the possibility to apply a reweighting of the multirate factors). 
    As a composition method, applying the reweighting will result in an order reduction. 
    Thus, order four is only achieved for M = 1 or reweighting=False such that a constant multirate factor is applied.

    Input:
    x0 : initial value x_0 = (p_0,q_0) of the state consisting of the momenta p, and the generalized coordinates q
    tspan : time interval [t0,t_end] on which the numerical solution will be computed.
    nsteps : number of time steps of constant step size h=(tspan[1]-tspan[0])/nsteps, i.e., we consider an equidistant time grid 
    M : multirate factor -- the fast part of the potential and the kinetic part will be integrated by applying M steps of the splitting method 
    reweighting: Boolean -- If true, the multirate factor M will be reweighted. Otherwise, M is kept constant.

    Output:
    t : time grid 
    x : 2d array of size 2d x (nsteps+1) where x[:,n] contains the numerical approximation for the time point t[n]
    """

    h = (tspan[1] - tspan[0])/nsteps
    t = np.linspace(tspan[0], tspan[1], nsteps+1)
    x = np.zeros((len(x0), nsteps+1))
    x[:,0] = x0
    d = len(x0)//2

    # composition weights
    c = np.array([
        0.0829844064174052,
        0.233995250731498,
        -0.409933719901930,
        0.059762097006575,
        0.370877414979582,
        0.162314550766870
    ])
    s = len(c)

    for n in range(nsteps):
        x[:,n+1] = x[:,n]

        for i in range(s):
            if reweighting: # potentially apply the reweighting of the multirate factor
                mrfactor = int(np.ceil(np.abs(c[i])*M))
            else:
                mrfactor = M

            
            for m in range(mrfactor): 
                x[8:,n+1] += c[i]*h/mrfactor * dTdp(x[:8,n+1])
                x[:8,n+1] += c[i]*h/mrfactor * f_fast(x[8:,n+1]) 

            x[:8,n+1] += (c[i]+c[s-1-i])*h*f_slow(x[8:,n+1]) # combination of the last step of the Lie-Trotter splitting and the first step of its adjoint

            if reweighting: # potentially apply the reweighting of the multirate factor
                mrfactor = int(np.ceil(np.abs(c[s-1-i])*M))
            else:
                mrfactor = M

            for m in range(mrfactor):
                x[:8,n+1] += c[s-1-i]*h/mrfactor * f_fast(x[8:,n+1])
                x[8:,n+1] += c[s-1-i]*h/mrfactor * dTdp(x[:8,n+1])
    return t,x

The hierarchical splitting approach also facilitates the derivation of splitting methods for $N$-split systems of order $p>4$. Exemplarily, we will design a sixth-order hierarchical splitting method. For comparison, we implement the ten-stage sixth-order splitting method from [Blanes & Moan 2002]. 

In [ ]:
def BMs10p6(x0,tspan,nsteps,M,reweighting=False):
    """ 
    Ten-stage composition method of order six from [Blanes & Moan 2002] 
    ---
    Sixth-order splitting method that is a composition of the Lie-Trotter splitting and its adjoint.

    The method is extended by a multiple time stepping approach (incl. the possibility to apply a reweighting of the multirate factors). 
    As a composition method, applying the reweighting will result in an order reduction. 
    Thus, order four is only achieved for M = 1 or reweighting=False such that a constant multirate factor is applied.

    Input:
    x0 : initial value x_0 = (p_0,q_0) of the state consisting of the momenta p, and the generalized coordinates q
    tspan : time interval [t0,t_end] on which the numerical solution will be computed.
    nsteps : number of time steps of constant step size h=(tspan[1]-tspan[0])/nsteps, i.e., we consider an equidistant time grid 
    M : multirate factor -- the fast part of the potential and the kinetic part will be integrated by applying M steps of the splitting method 
    reweighting: Boolean -- If true, the multirate factor M will be reweighted. Otherwise, M is kept constant.

    Output:
    t : time grid 
    x : 2d array of size 2d x (nsteps+1) where x[:,n] contains the numerical approximation for the time point t[n]
    """

    h = (tspan[1] - tspan[0])/nsteps
    t = np.linspace(tspan[0], tspan[1], nsteps+1)
    x = np.zeros((len(x0), nsteps+1))
    x[:,0] = x0
    d = len(x0)//2

    # composition weights
    c = np.array([
    0.0985536835006498,
    -0.44734648269547816,
    -0.42511876779769087,
    0.19560248860005314,
    -0.36276277925434486,
    0.34635818985072686,
    0.23706391397812188,
    0.49242637248987586,
    0.31496061692769417,
    0.05026276440039221
    ])
    s = len(c)

    for n in range(nsteps):
        x[:,n+1] = x[:,n]

        for i in range(s):
            if reweighting: # potentially apply the reweighting of the multirate factor
                mrfactor = int(np.ceil(np.abs(c[s-1-i])*M))
            else:
                mrfactor = M

            for m in range(mrfactor):
                x[8:,n+1] += c[s-1-i]*h/mrfactor * dTdp(x[:8,n+1])
                x[:8,n+1] += c[s-1-i]*h/mrfactor * f_fast(x[8:,n+1]) 

            x[:8,n+1] += (c[i]+c[s-1-i])*h*f_slow(x[8:,n+1]) # combination of the last step of the Lie-Trotter splitting and the first step of its adjoint
            
            if reweighting: # potentially apply the reweighting of the multirate factor
                mrfactor = int(np.ceil(np.abs(c[i])*M))
            else:
                mrfactor = M
            
            for m in range(mrfactor):
                x[:8,n+1] += c[i]*h/mrfactor * f_fast(x[8:,n+1])
                x[8:,n+1] += c[i]*h/mrfactor * dTdp(x[:8,n+1])
    return t,x

As a hierarchical splitting method, we implement a sixth-order method that assigns the splitting method `BABABABABABABAB` from [Omelyan et al. 2003] to both inner nodes.

In [ ]:
def s8p6_core_step(p,q,c,h,M,reweighting):
    """ 
    Inner step of the Hs8p6 method which computes M time steps (with potential reweighting of the multirate factor) 
    of the s8p6/BABABABABABABAB splitting method from [Omelyan et al. 2003] for the kinetic part and the fast part of the potential.  

    Input:
    p : current state of the momenta 
    q : current state of the coordinates 
    c : fractional time step, i.e., the function aims to advance the numerical solution by a step of size c*h
    h : macro time step
    M : multirate factor
    reweighting: Boolean. If True, then the multirate factor is reweighted, otherwise a constant multirate factor M is utilized

    Output:
    p_new : updated momenta
    q_new : updated coordinates 
    """

    # integrator coefficients
    a2 = 0.2465881872786138
    a3 = 0.6047073875057809
    a4 = -0.4009869039788007
    a5 = 1-2*(a2+a3+a4)
    b1 = 0.0833333333333333
    b2 = 0.3977675859548440
    b3 = -0.0393336931446257
    b4 = 0.5-(b1+b2+b3)

    q_new = q.copy()
    p_new = p.copy()

    if reweighting: # potentially apply the reweighting of the multirate factor
        M = int(np.ceil(np.abs(c)*M))

    val = f_fast(q_new) # since the first momentum update of iteration m+1 uses the 
                        # same force evaluation as the last momentum update of iteration m, 
                        # we store the result of the force evaluation to re-use it.

    for m in range(M):
        p_new += c*h*b1/M * val # momentum update
        q_new += c*h*a2/M * dTdp(p_new) # position update 
        p_new += c*h*b2/M * f_fast(q_new)
        q_new += c*h*a3/M * dTdp(p_new)
        p_new += c*h*b3/M * f_fast(q_new)
        q_new += c*h*a4/M * dTdp(p_new)
        p_new += c*h*b4/M * f_fast(q_new)
        q_new += c*h*a5/M * dTdp(p_new)
        p_new += c*h*b4/M * f_fast(q_new)
        q_new += c*h*a4/M * dTdp(p_new)
        p_new += c*h*b3/M * f_fast(q_new)
        q_new += c*h*a3/M * dTdp(p_new)
        p_new += c*h*b2/M * f_fast(q_new)
        q_new += c*h*a2/M * dTdp(p_new)
        val = f_fast(q_new)
        p_new += c*h*b1/M * val

    return p_new,q_new

def Hierarchicals8p6(x0,tspan,nsteps,M,reweighting=False):
    """ 
    Hs8p6 method 
    ---
    Sixth-order hierarchical splitting method for separable Hamiltonian systems of the form H(p,q) = T(p) + V_S(q) + V_F(q).
    The method uses the s8p6/BABABABABABABAB splitting method from [Omelyan et al. 2003] in both inner nodes, and the exact flows in all leaf nodes.

    The implementation includes a multiple time stepping approach (incl. the possibility to apply a reweighting of the multirate factors 
    to reduce computational overhead).

    Input:
    x0 : initial value x_0 = (p_0,q_0) of the state consisting of the momenta p, and the generalized coordinates q
    tspan : time interval [t0,t_end] on which the numerical solution will be computed.
    nsteps : number of time steps of constant step size h=(tspan[1]-tspan[0])/nsteps, i.e., we consider an equidistant time grid 
    M : multirate factor -- the fast part of the potential and the kinetic part will be integrated by applying M steps of the splitting method 
    reweighting: Boolean -- If true, the multirate factor M will be reweighted. Otherwise, M is kept constant.

    Output:
    t : time grid 
    x : 2d array of size 2d x (nsteps+1) where x[:,n] contains the numerical approximation for the time point t[n]
    """

    h = (tspan[1] - tspan[0])/nsteps
    t = np.linspace(tspan[0], tspan[1], nsteps+1)
    x = np.zeros((len(x0), nsteps+1))
    x[:,0] = x0
    d = len(x0)//2

    # integrator coefficients
    a2 = 0.2465881872786138
    a3 = 0.6047073875057809
    a4 = -0.4009869039788007
    a5 = 1-2*(a2+a3+a4)
    b1 = 0.0833333333333333
    b2 = 0.3977675859548440
    b3 = -0.0393336931446257
    b4 = 0.5-(b1+b2+b3)
    
    val = f_slow(x[8:,0]) # since the first momentum update of iteration n+1 uses the 
                          # same force evaluation as the last momentum update of iteration n, 
                          # we store the result of the force evaluation to re-use it.

    for n in range(nsteps):
        x[:,n+1] = x[:,n]
        x[:8,n+1] += h*b1*val # momentum update employing the slow part of the force
        x[:8,n+1],x[8:,n+1] = s8p6_core_step(x[:8,n+1],x[8:,n+1],a2,h,M,reweighting) # fast subsystem computed by another call of the s8p6 splitting method
        x[:8,n+1] += h*b2*f_slow(x[8:,n+1]) 
        x[:8,n+1],x[8:,n+1] = s8p6_core_step(x[:8,n+1],x[8:,n+1],a3,h,M,reweighting)
        x[:8,n+1] += h*b3*f_slow(x[8:,n+1])
        x[:8,n+1],x[8:,n+1] = s8p6_core_step(x[:8,n+1],x[8:,n+1],a4,h,M,reweighting)
        x[:8,n+1] += h*b4*f_slow(x[8:,n+1]) 
        x[:8,n+1],x[8:,n+1] = s8p6_core_step(x[:8,n+1],x[8:,n+1],a5,h,M,reweighting)
        x[:8,n+1] += h*b4*f_slow(x[8:,n+1]) 
        x[:8,n+1],x[8:,n+1] = s8p6_core_step(x[:8,n+1],x[8:,n+1],a4,h,M,reweighting)
        x[:8,n+1] += h*b3*f_slow(x[8:,n+1]) 
        x[:8,n+1],x[8:,n+1] = s8p6_core_step(x[:8,n+1],x[8:,n+1],a3,h,M,reweighting)
        x[:8,n+1] += h*b2*f_slow(x[8:,n+1]) 
        x[:8,n+1],x[8:,n+1] = s8p6_core_step(x[:8,n+1],x[8:,n+1],a2,h,M,reweighting)
        val = f_slow(x[8:,n+1])
        x[:8,n+1] += h*b1*val
    return t,x

## Numerical Results

In [ ]:
t_ref,x_ref = Hierarchicals8p6(x0,tspan,tspan[1]*1250,7,True) # reference solution 
x_eval = x_ref[:,-1]

### 1. Results for the `HOMF4` method

In [ ]:
NIT = 10

nstep_array = tspan[1]*np.array([64,100,150,200,300,400,500])
global_error = np.zeros((3,len(nstep_array)))
CPU_times = np.zeros((3,len(nstep_array)))

for i in range(len(nstep_array)):
    print(i)
    
    for _ in range(NIT):
        start = time.process_time()
        t,x = HierarchicalOMF4(x0,tspan,nstep_array[i],15,True)
        end = time.process_time()
        CPU_times[0,i] += end-start
    CPU_times[0,i] /= NIT
    global_error[0,i] = np.linalg.norm(x_eval - x[:,-1])

    for _ in range(NIT):
        start = time.process_time()
        t,x = HierarchicalOMF4(x0,tspan,nstep_array[i],9,False) # 9 = ceil(0.556648713623280 * 15)
        end = time.process_time()
        CPU_times[1,i] += end-start
    CPU_times[1,i] /= NIT
    global_error[1,i] = np.linalg.norm(x_eval - x[:,-1])

    for _ in range(NIT):
        start = time.process_time()
        t,x = HierarchicalOMF4(x0,tspan,8*nstep_array[i],1,False)
        end = time.process_time()
        CPU_times[2,i] += end-start
    CPU_times[2,i] /= NIT
    global_error[2,i] = np.linalg.norm(x_eval - x[:,-1])

# store the results
np.savez('HierarchicalOMF4_results.npz',  
         nstep_array=nstep_array, 
         global_error=global_error, 
         CPU_times=CPU_times)
print("results saved in 'HierarchicalOMF4_results.npz'")

In [ ]:
# load the results
data = np.load('HierarchicalOMF4_results.npz')
nstep_array = data['nstep_array']
global_error = data['global_error']
CPU_times = data['CPU_times']

plt.loglog(CPU_times[0,:],global_error[0,:],marker='o',label=f'HierarchicalOMF4 (reweighted M = 15)')
plt.loglog(CPU_times[1,:],global_error[1,:],marker='o',label=f'HierarchicalOMF4 (constant M = 9)')
plt.loglog(CPU_times[2,:],global_error[2,:],marker='o',label=f'HierarchicalOMF4 (M = 1)')
plt.legend()
plt.xlabel('CPU time [s]')
plt.ylabel('Global error at t = 40')
plt.show()

In [ ]:
# load the results
data = np.load('HierarchicalOMF4_results.npz')
nstep_array = data['nstep_array']
global_error = data['global_error']
CPU_times = data['CPU_times']

plt.loglog(CPU_times[0,:],global_error[0,:],marker='o',label=f'HierarchicalOMF4 (reweighted M = 15)')
plt.loglog(CPU_times[1,:],global_error[1,:],marker='o',label=f'HierarchicalOMF4 (constant M = 9)')
plt.legend()
plt.xlabel('CPU time [s]')
plt.ylabel('Global error at t = 40')
matplot2tikz.save("4body_HOMF4.tex")

### 2. Results for the `COMP4` method

In [ ]:
NIT = 10
nstep_array = tspan[1]*np.array([64,100,150,200,300,400,500])
M_array = np.array([100,200,300,400])
global_error = np.zeros((len(M_array)+3,len(nstep_array)))
CPU_times = np.zeros((len(M_array)+3,len(nstep_array)))

for i in range(len(nstep_array)):
    print(i)
    for j in range(len(M_array)):
        for _ in range(NIT):
            start = time.process_time()
            t,x = COMP4(x0,tspan,nstep_array[i],M_array[j],True)
            end = time.process_time()
            CPU_times[j,i] += end-start
        CPU_times[j,i] /= NIT
        global_error[j,i] = np.linalg.norm(x_eval - x[:,-1])

    for _ in range(NIT):
        start = time.process_time()
        t,x = COMP4(x0,tspan,nstep_array[i],nstep_array[i]/tspan[1],True)
        end = time.process_time()
        CPU_times[-3,i] += end-start
    CPU_times[-3,i] /= NIT
    global_error[-3,i] = np.linalg.norm(x_eval - x[:,-1])

    for _ in range(NIT):
        start = time.process_time()
        t,x = COMP4(x0,tspan,nstep_array[i],223,False) # ceil(0.5566487136232801 * 400) = 223
        end = time.process_time()
        CPU_times[-2,i] += end-start
    CPU_times[-2,i] /= NIT
    global_error[-2,i] = np.linalg.norm(x_eval - x[:,-1])

    for _ in range(NIT):
        start = time.process_time()
        t,x = COMP4(x0,tspan,30*nstep_array[i],1,False)
        end = time.process_time()
        CPU_times[-1,i] += end-start
    CPU_times[-1,i] /= NIT
    global_error[-1,i] = np.linalg.norm(x_eval - x[:,-1])

# store the results
np.savez('COMP4_results.npz',  
         nstep_array=nstep_array, 
         M_array=M_array,
         global_error=global_error, 
         CPU_times=CPU_times)
print("results saved in 'COMP4_results.npz'")

In [ ]:
# load the results
data = np.load('COMP4_results.npz')
nstep_array = data['nstep_array']
M_array = data['M_array']
global_error = data['global_error']
CPU_times = data['CPU_times']

for j in range(len(M_array)):
    plt.loglog(tspan[1]/nstep_array,global_error[j,:],marker='o',label=f'COMP4 (M = {M_array[j]}, reweighting)')
plt.loglog(tspan[1]/nstep_array,global_error[-3,:],marker='o',label=f'COMP4 (M = 1/h, reweighting)')
plt.legend()
plt.xlabel('Step size h')
plt.ylabel('Global error at t = 40')
matplot2tikz.save("4body_COMP4.tex")

In [ ]:
# load the results
data = np.load('COMP4_results.npz')
nstep_array = data['nstep_array']
global_error = data['global_error']
CPU_times = data['CPU_times']

plt.loglog(CPU_times[-4,:],global_error[-4,:],marker='o',label=f'COMP4 (reweighted M = 400)')
plt.loglog(CPU_times[-2,:],global_error[-2,:],marker='o',label=f'COMP4 (constant M = 223)')
plt.loglog(CPU_times[-1,:],global_error[-1,:],marker='o',label=f'COMP4 (M = 1)')
plt.legend()
plt.xlabel('CPU time [s]')
plt.ylabel('Global error at t = 40')
plt.show()

### 3. Results for the `BMs6p4` method

In [ ]:
NIT = 10

nstep_array = tspan[1]*2*np.array([64,100,150,200,300,400,500,750,1000,1500])
M_array = np.array([3])
global_error = np.zeros((2*len(M_array),len(nstep_array)))
CPU_times = np.zeros((2*len(M_array),len(nstep_array)))

for i in range(len(nstep_array)):
    print(i)
    for j in range(len(M_array)):
        for _ in range(NIT):
            start = time.process_time()
            t,x = BMs6p4(x0,tspan,nstep_array[i],3,False)
            end = time.process_time()
            CPU_times[j,i] += end-start
        CPU_times[j,i] /= NIT
        global_error[j,i] = np.linalg.norm(x_eval - x[:,-1])

    for j in range(len(M_array)):
        for _ in range(NIT):
            start = time.process_time()
            t,x = BMs6p4(x0,tspan,6*nstep_array[i],3,True)
            end = time.process_time()
            CPU_times[len(M_array)+j,i] += end-start
        CPU_times[len(M_array)+j,i] /= NIT
        global_error[len(M_array)+j,i] = np.linalg.norm(x_eval - x[:,-1])

# store the results
np.savez('BMs6p4_results.npz',  
         nstep_array=nstep_array, 
         global_error=global_error, 
         CPU_times=CPU_times)
print("results saved in 'BMs6p4_results.npz'")

In [ ]:
# load the results
data = np.load('BMs6p4_results.npz')
nstep_array = data['nstep_array']
global_error = data['global_error']
CPU_times = data['CPU_times']

plt.loglog(CPU_times[0,:],global_error[0,:],marker='o',label=f'BMs6p4 (constant M = 3)')
plt.loglog(CPU_times[1,:],global_error[1,:],marker='o',label=f'BMs6p4 (reweighted M = 3)')
plt.legend()
plt.xlabel('CPU time [s]')
plt.ylabel('Global error at t = 40')
matplot2tikz.save("4body_BMs6p4.tex")

### 4. Comparison of the results in a single work-precision diagram

In [ ]:
# load the results
data = np.load('HierarchicalOMF4_results.npz')
nstep_array = data['nstep_array']
global_error = data['global_error']
CPU_times = data['CPU_times']
plt.loglog(CPU_times[0,:],global_error[0,:],marker='o',label=f'HierarchicalOMF4 (reweighted M = 15)')

# load the results
data = np.load('COMP4_results.npz')
nstep_array = data['nstep_array']
global_error = data['global_error']
CPU_times = data['CPU_times']
plt.loglog(CPU_times[3,:],global_error[3,:],marker='s',label=f'COMP4 (reweighted M = 400)')

# load the results
data = np.load('BMs6p4_results.npz')
nstep_array = data['nstep_array']
global_error = data['global_error']
CPU_times = data['CPU_times']
plt.loglog(CPU_times[0,:],global_error[0,:],marker='^',label=f'BMs6p4 (constant M = 3)')

plt.xlabel('CPU Time [s]')
plt.ylabel('Global Error at t = 40')
plt.legend()
matplot2tikz.save("4body_4th_order-comparison.tex")

### 5. Order-6 Tests

In [ ]:
t_ref,x_ref = Hierarchicals8p6(x0,tspan,tspan[1]*1250,7,True) # reference solution 
x_eval = x_ref[:,-1]

In [ ]:
NIT = 10

nstep_array = tspan[1]*np.array([64,100,150,200,300,400,500])
global_error = np.zeros((2,len(nstep_array)))
CPU_times = np.zeros((2,len(nstep_array)))

for i in range(len(nstep_array)):
    print(i)
    for _ in range(NIT):
        start = time.process_time()
        t,x = BMs10p6(x0,tspan,nstep_array[i],3,False)
        end = time.process_time()
        CPU_times[0,i] += end-start
    CPU_times[0,i] /= NIT
    global_error[0,i] = np.linalg.norm(x_eval - x[:,-1])

    for _ in range(NIT):
        start = time.process_time()
        t,x = BMs10p6(x0,tspan,30*nstep_array[i],3,True)
        end = time.process_time()
        CPU_times[1,i] += end-start
    CPU_times[1,i] /= NIT
    global_error[1,i] = np.linalg.norm(x_eval - x[:,-1])

# store the results
np.savez('BMs10p6_results.npz',  
         nstep_array=nstep_array, 
         global_error=global_error, 
         CPU_times=CPU_times)
print("results saved in 'BMs10p6_results.npz'")

In [ ]:
# load the results
data = np.load('BMs10p6_results.npz')
nstep_array = data['nstep_array']
global_error = data['global_error']
CPU_times = data['CPU_times']

plt.loglog(CPU_times[0,:],global_error[0,:],marker='o',label=f'BMs10p6 (M = 3, constant)')
plt.loglog(CPU_times[1,:],global_error[1,:],marker='o',label=f'BMs10p6 (M = 3, reweighted)')
plt.legend()
plt.xlabel('CPU Time [s]')      
plt.ylabel('Global Error at t = 40')        
matplot2tikz.save("4body_BMs10p6.tex")

In [ ]:
NIT = 10

nstep_array = tspan[1]*np.array([20,75,100,125,150,175,200,250])
global_error = np.zeros((2,len(nstep_array)))
CPU_times = np.zeros((2,len(nstep_array)))

for i in range(len(nstep_array)):
    print(i)
    for _ in range(NIT):
        start = time.process_time()
        t,x = Hierarchicals8p6(x0,tspan,nstep_array[i],7,True)
        end = time.process_time()
        CPU_times[0,i] += end-start
    CPU_times[0,i] /= NIT
    global_error[0,i] = np.linalg.norm(x_eval - x[:,-1])

    for _ in range(NIT):
        start = time.process_time()
        t,x = Hierarchicals8p6(x0,tspan,nstep_array[i],5,False) # ceil(0.6047073875057809 * 7) = 5
        end = time.process_time()
        CPU_times[1,i] += end-start
    CPU_times[1,i] /= NIT
    global_error[1,i] = np.linalg.norm(x_eval - x[:,-1])

# store the results
np.savez('Hierarchicals8p6_results.npz',  
         nstep_array=nstep_array, 
         global_error=global_error, 
         CPU_times=CPU_times)
print("results saved in 'Hierarchicals8p6_results.npz'")

In [ ]:
# load the results
data = np.load('Hierarchicals8p6_results.npz')
nstep_array = data['nstep_array']
global_error = data['global_error']
CPU_times = data['CPU_times']

plt.loglog(CPU_times[0,:],global_error[0,:],marker='o',label=f'Hs8p6 (M = 7, reweighted)')
plt.loglog(CPU_times[1,:],global_error[1,:],marker='o',label=f'Hs8p6 (M = 5, constant)')
plt.legend()
plt.xlabel('CPU Time [s]')      
plt.ylabel('Global Error at t = 40')        
plt.show()

In [ ]:
# load the results
data = np.load('Hierarchicals8p6_results.npz')
nstep_array = data['nstep_array']
global_error = data['global_error']
CPU_times = data['CPU_times']

plt.loglog(CPU_times[0,:],global_error[0,:],marker='o',label=f'Hs8p6 (M = 7, reweighted)')

# load the results
data = np.load('BMs10p6_results.npz')
nstep_array = data['nstep_array']
global_error = data['global_error']
CPU_times = data['CPU_times']

plt.loglog(CPU_times[0,:],global_error[0,:],marker='o',label=f'BMs10p6 (M = 3, constant)')

plt.legend()
plt.xlabel('CPU Time [s]')
plt.ylabel('Global Error at t = 40')
matplot2tikz.save("4body_6th_order-comparison.tex")

In [ ]:
data = np.load('HierarchicalOMF4_results.npz')
global_error = data['global_error']
CPU_times = data['CPU_times']

m1, b1 = np.polyfit(np.log(CPU_times[0,1:-1]), np.log(global_error[0,1:-1]), 1)
HOMF4_fit = lambda y: (y-b1)/m1

data = np.load('BMs6p4_results.npz')
global_error = data['global_error']
CPU_times = data['CPU_times']

m2, b2 = np.polyfit(np.log(CPU_times[0,1:-1]), np.log(global_error[0,1:-1]), 1)
BMs6p4_fit = lambda y: (y-b2)/m2

data = np.load('Hierarchicals8p6_results.npz')
global_error = data['global_error']
CPU_times = data['CPU_times']

m3, b3 = np.polyfit(np.log(CPU_times[0,1:-1]), np.log(global_error[0,1:-1]), 1)
Hs8p6_fit = lambda y: (y-b3)/m3

data = np.load('BMs10p6_results.npz')
global_error = data['global_error']
CPU_times = data['CPU_times']

m4, b4 = np.polyfit(np.log(CPU_times[0,1:-1]), np.log(global_error[0,1:-1]), 1)
BMs10p6_fit = lambda y: (y-b4)/m4

TOL = 1e-6 # error tolerance for comparison

print( np.exp(HOMF4_fit(np.log(TOL)))/np.exp(BMs6p4_fit(np.log(TOL))) )
print( np.exp(Hs8p6_fit(np.log(TOL)))/np.exp(BMs10p6_fit(np.log(TOL))) )

## Force-gradient approach

In [ ]:
def hessian_V_S(q):
    
    H = np.zeros((8, 8))
    
    I = np.eye(2)
    
    for i in range(4):
        for j in range(4):
            if i < j:
                if i != 1 or j != 3:
                    rij = q[2*i:2*i+2] - q[2*j:2*j+2]
                    dist = np.linalg.norm(rij)
                    
                    outer = np.outer(rij, rij)
                    
                    block = (1 / dist**3) * I - (3 / dist**5) * outer
                    block *= m[i] * m[j]
                    
                    # diagonal blocks
                    H[2*i:2*i+2, 2*i:2*i+2] += block
                    H[2*j:2*j+2, 2*j:2*j+2] += block
                    
                    # off-diagonal blocks
                    H[2*i:2*i+2, 2*j:2*j+2] -= block
                    H[2*j:2*j+2, 2*i:2*i+2] -= block
    
    return H

def hessian_V_F(q):
    
    H = np.zeros((8, 8))
    
    I = np.eye(2)
    
    rij = q[2*1:2*1+2] - q[2*3:2*3+2]
    dist = np.linalg.norm(rij)
    
    outer = np.outer(rij, rij)
    
    block = (1 / dist**3) * I - (3 / dist**5) * outer
    block *= m[1] * m[3]
    
    # diagonal blocks
    H[2*1:2*1+2, 2*1:2*1+2] += block
    H[2*3:2*3+2, 2*3:2*3+2] += block
    
    # off-diagonal blocks
    H[2*1:2*1+2, 2*3:2*3+2] -= block
    H[2*3:2*3+2, 2*1:2*1+2] -= block
    
    return H

def fg_slow(q):
    return 2*hessian_V_S(q) @ Minv @ f_slow(q) 

def fg_fast(q):
    return 2*hessian_V_F(q) @ Minv @ f_fast(q)

In [ ]:
def FGI_core_step(p,q,c,h,M,reweighting):
    """
    Hierarchical force-gradient integrator based on BACAB
    """
    q_new = q.copy()
    p_new = p.copy()

    if reweighting:
        M = int(np.ceil(np.abs(c)*M))

    for m in range(M):
        p_new += c*h/(6*M) * f_fast(q_new)
        q_new += c*h/(2*M) * dTdp(p_new)
        p_new += 2*c*h/(3*M) * f_fast(q_new) - 1/72 * (c*h/M)**3 * fg_fast(q_new)
        q_new += c*h/(2*M) * dTdp(p_new)
        p_new += c*h/(6*M) * f_fast(q_new)

    return p_new,q_new

def HierarchicalFGI(x0,tspan,nsteps,M,reweighting=True):
    """
    Hierarchical Hessian-free force-gradient integrator based on BADAB
    """
    h = (tspan[1] - tspan[0])/nsteps
    t = np.linspace(tspan[0], tspan[1], nsteps+1)
    x = np.zeros((len(x0), nsteps+1))
    x[:,0] = x0
    d = len(x0)//2

    for n in range(nsteps):
        x[:,n+1] = x[:,n]

        x[:8,n+1] += (h/6)*f_slow(x[8:,n+1])
        x[:8,n+1],x[8:,n+1] = FGI_core_step(x[:8,n+1],x[8:,n+1],0.5,h,M,reweighting)
        x[:8,n+1] += (2*h/3)*f_slow(x[8:,n+1]) -(1/72)*h**3*fg_slow(x[8:,n+1])
        x[:8,n+1],x[8:,n+1] = FGI_core_step(x[:8,n+1],x[8:,n+1],0.5,h,M,reweighting)
        x[:8,n+1] += (h/6)*f_slow(x[8:,n+1])

    return t,x